# Grade

Notebook 02 showed the observable features explaining 27% of the variance in interest rate. This notebook now asks where the rest is. How much does `loan_grade` explain and can `loan_grade`be derived from the other features?

## Setup

In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.data import load_clean
from src.features import build_preprocessor, split_xy, GRADES
from src.evaluate import cv_report, CV

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_val_score


## Comparing With and Without Grade

In [2]:
df = load_clean()
X_obs, y = split_xy(df)                              # grade dropped
X_grade, _ = split_xy(df, drop=["loan_status"])      # grade kept

grade_only = ColumnTransformer([("grade", OrdinalEncoder(categories=[GRADES]), ["loan_grade"])])

obs_table = cv_report(
    {"observables only": make_pipeline(build_preprocessor(), LinearRegression())},
    X_obs, y)

grade_table = cv_report(
    {"grade only":          make_pipeline(grade_only, LinearRegression()),
     "observables + grade": make_pipeline(build_preprocessor(include_grade=True), LinearRegression())},
    X_grade, y)

pd.concat([obs_table, grade_table])

,r2,r2_std,rmse,mae
model,,,,
observables only,0.2739,0.0067,2.7516,2.2109
grade only,0.8713,0.0038,1.1581,0.9033
observables + grade,0.8721,0.0039,1.1547,0.9036


Including grade changes everything: $R^2$ goes from 0.27 on the observables to 0.87 on grade alone, matching the staircase in the EDA boxplot. Adding the observables on top of grade gains only about 0.001 (from 0.871 to 0.872; MAE identical at 0.90), so the observable features contain almost no information about the rate that grade doesn't already encode. The rate is determined through grade.

## Can grade be predicted from the observable features?

In [3]:
clf = make_pipeline(build_preprocessor(), LogisticRegression(max_iter=2000))
acc = cross_val_score(clf, X_obs, df["loan_grade"], cv=CV, scoring="accuracy")
print(f"accuracy {acc.mean():.3f} ± {acc.std():.3f}   (majority class: {df.loan_grade.value_counts(normalize=True).max():.3f})")

accuracy 0.468 ± 0.005   (majority class: 0.328)


In [4]:
df["loan_grade"].value_counts(normalize=True).sort_index().round(3)

loan_grade
A    0.328
B    0.320
C    0.199
D    0.113
E    0.030
F    0.007
G    0.002
Name: proportion, dtype: float64

Grades are heavily skewed toward the left (A and B are 65% of rows, F and G together under 1%), so the classifier mostly separates the common grades, and the rare high-rate grades are very hard to learn from this data. The logistic regression's 47% accuracy beats always predicting the most common grade (A: 33%), so the observables carry some information about grade, but far from all of it. Grade, and largely with it the rate, is likely set from information outside this dataset.

## Conclusion

Grade nearly determines the rate (grade-only MAE: 0.90), and adding the observables does not improve on it (grade + observables MAE: 0.90), so they hold no measurable additional information about the rate. The logistic regression recovers grade with 47% accuracy versus 33% for always predicting the most common grade (A). The observables partially predict grade, but not enough to reproduce it. Overall, the dataset's observable features are not sufficient to predict interest rates well.